In [23]:
import pandas as pd

1. Forecasted Demand Distribution
    - Using clustering output:Hub, Shift, Employees
2. Vehicle Capacity Analysis
    - For each group: Example: Saravanampatti 04:30, 177 employees    
    - How many: SEDAN, SUV, VAN are needed?
3. Utilization Analysis
    - Example:177 employees, 22 Vans = 176 seats, Utilization = 100.6%
    - 23 Vans = 184 seats, Utilization = 96.2%
4. Cost Analysis
    - Using pricing_engine.    
5. Escort Analysis
    - We discovered: 368 escort-required employees, That's huge. This may affect: Vehicle type, Vendor selection, Cost    

Optimize: Cost + Utilization + Safety + Vendor Quality

EDA ---> prototype ---> allocation strategy decision

# Business Objective
Goal:

Given forecasted employee demand,
determine:

1. Vehicle count
2. Vehicle type mix
3. Capacity utilization
4. Estimated transport cost
5. Escort requirement
6. Vendor requirement

while balancing:

- Cost
- Utilization
- Safety
- SLA

In [24]:
# load fleet registry
fleet_df = pd.read_csv(
    r"D:\mlprojects\trns_proj\data\fleet_registry.csv"
)

fleet_df.head()

,vehicle_id,vendor_name,vehicle_type,seating_capacity,fuel_type,gps_enabled,panic_button_enabled,safety_rating,active
0,FT-SDN-001,FastTrack Mobility,TEMPO_TRAVELLER,14,DIESEL,True,True,8.8,True
1,SR-SUV-101,SecureRide Logistics,SUV,6,DIESEL,True,True,9.5,True
2,CC-VAN-220,CityCommute Services,MAZDA,20,CNG,True,False,8.1,True


In [25]:
fleet_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   vehicle_id            3 non-null      str    
 1   vendor_name           3 non-null      str    
 2   vehicle_type          3 non-null      str    
 3   seating_capacity      3 non-null      int64  
 4   fuel_type             3 non-null      str    
 5   gps_enabled           3 non-null      bool   
 6   panic_button_enabled  3 non-null      bool   
 7   safety_rating         3 non-null      float64
 8   active                3 non-null      bool   
dtypes: bool(3), float64(1), int64(1), str(4)
memory usage: 411.0 bytes


In [26]:
fleet_df.describe(
    include="all"
)

,vehicle_id,vendor_name,vehicle_type,seating_capacity,fuel_type,gps_enabled,panic_button_enabled,safety_rating,active
count,3,3,3,3.000000,3,3,3,3.00,3
unique,3,3,3,NaN,2,1,2,NaN,1
top,FT-SDN-001,FastTrack Mobility,TEMPO_TRAVELLER,NaN,DIESEL,True,True,NaN,True
freq,1,1,1,NaN,2,3,2,NaN,3
mean,NaN,NaN,NaN,13.333333,NaN,NaN,NaN,8.80,NaN
std,NaN,NaN,NaN,7.023769,NaN,NaN,NaN,0.70,NaN
min,NaN,NaN,NaN,6.000000,NaN,NaN,NaN,8.10,NaN
25%,NaN,NaN,NaN,10.000000,NaN,NaN,NaN,8.45,NaN
50%,NaN,NaN,NaN,14.000000,NaN,NaN,NaN,8.80,NaN
75%,NaN,NaN,NaN,17.000000,NaN,NaN,NaN,9.15,NaN


## Fleet Inventory EDA
How many vehicles?

How many sedans?

How many SUVs?

How many vans?

Available capacity?

In [27]:
fleet_df[
    "vehicle_type"
].value_counts()

vehicle_type
TEMPO_TRAVELLER    1
SUV                1
MAZDA              1
Name: count, dtype: int64

In [28]:
fleet_df.groupby(
    "vehicle_type"
)[
    "seating_capacity"
].agg(
    [
        "count",
        "mean",
        "sum",
    ]
)

,count,mean,sum
vehicle_type,,,
MAZDA,1,20.0,20
SUV,1,6.0,6
TEMPO_TRAVELLER,1,14.0,14


In [29]:
fleet_df[
    "vendor_name"
].value_counts()

vendor_name
FastTrack Mobility      1
SecureRide Logistics    1
CityCommute Services    1
Name: count, dtype: int64

In [30]:
pd.crosstab(
    fleet_df["vendor_name"],
    fleet_df["vehicle_type"]
)

vehicle_type,MAZDA,SUV,TEMPO_TRAVELLER
vendor_name,,,
CityCommute Services,1,0,0
FastTrack Mobility,0,0,1
SecureRide Logistics,0,1,0


In [31]:
demand_df = pd.DataFrame(
    {
        "hub": [
            "Ganapathy",
            "Ganapathy",
            "Hopes",
            "Hopes",
            "Saravanampatti",
            "Saravanampatti",
            "Singanallur",
            "Singanallur",
            "Thudiyalur",
            "Thudiyalur",
        ],

        "transport_shift": [
            "03:30",
            "04:30",
            "03:30",
            "04:30",
            "03:30",
            "04:30",
            "03:30",
            "04:30",
            "03:30",
            "04:30",
        ],

        "employee_count": [
            33,
            55,
            60,
            79,
            126,
            177,
            73,
            141,
            77,
            75,
        ]
    }
)

In [ ]:
# if use mazda only Capacity = 20
import math

demand_df["van_count"] = (
    demand_df["employee_count"]
    .apply(
        lambda x:
        math.ceil(x / 20)
    )
)

demand_df["available_seats"] = (
    demand_df["van_count"] * 8
)

demand_df["utilization_pct"] = (
    demand_df["employee_count"]
    /
    demand_df["available_seats"]
) * 100

demand_df

,hub,transport_shift,employee_count,van_count,available_seats,utilization_pct
0,Ganapathy,03:30,33,5,40,82.500000
1,Ganapathy,04:30,55,7,56,98.214286
2,Hopes,03:30,60,8,64,93.750000
3,Hopes,04:30,79,10,80,98.750000
4,Saravanampatti,03:30,126,16,128,98.437500
5,Saravanampatti,04:30,177,23,184,96.195652
6,Singanallur,03:30,73,10,80,91.250000
7,Singanallur,04:30,141,18,144,97.916667
8,Thudiyalur,03:30,77,10,80,96.250000
9,Thudiyalur,04:30,75,10,80,93.750000


- for van only strategy average utilization is approximately: 94-95% which is outstanding.